# Лабораторная работа 7. FCNN — Регрессия
**Датасет:** `cars_end.csv`, target=`price_usd`

## 1. Импорты

In [3]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import uniform

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import tensorflow as tf
import keras
from keras import layers
import keras_tuner as kt

import ray
from ray import tune as ray_tune
from ray.tune.search.optuna import OptunaSearch

print("TF:", tf.__version__, "| Keras:", keras.__version__, "| Ray:", ray.__version__)


TF: 2.16.2 | Keras: 3.14.1 | Ray: 2.55.1


## 2. Данные

In [4]:
df = pd.read_csv("../lab4/cars_end.csv")
X_raw = df.drop(columns=["price_usd"])
y = df["price_usd"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

sel = SelectKBest(f_regression, k=30)
X = sel.fit_transform(X_scaled, y)

print("Признаков после отбора:", X.shape[1])

SEED = 42
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
print("Train:", X_tr.shape, "| Test:", X_te.shape)


Признаков после отбора: 30
Train: (30792, 30) | Test: (7699, 30)


## 3. Scikit-learn MLP

### Вспомогательная функция

In [5]:
results = []

def eval_reg(model, label, optimizer):
    r = {}
    for tag, X, y in [("train", X_tr, y_tr), ("test", X_te, y_te)]:
        p = model.predict(X)
        r[tag] = {"R2": r2_score(y, p),
                  "MSE": mean_squared_error(y, p),
                  "MAE": mean_absolute_error(y, p)}
    print(f"{label:40s} | train R2={r['train']['R2']:.3f} | test R2={r['test']['R2']:.3f}")
    results.append({"Модель": label, "Оптимизатор": optimizer,
                    **{f"train_{k}": v for k, v in r["train"].items()},
                    **{f"test_{k}": v for k, v in r["test"].items()}})
    return r


### 3.1 Optuna

In [6]:
def objective(trial):
    m = MLPRegressor(
        hidden_layer_sizes=(trial.suggest_int("h1", 32, 256),
                            trial.suggest_int("h2", 16, 128)),
        activation=trial.suggest_categorical("activation", ["relu", "tanh"]),
        solver=trial.suggest_categorical("solver", ["adam", "lbfgs"]),
        alpha=trial.suggest_float("alpha", 1e-5, 1e-2, log=True),
        learning_rate_init=trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        max_iter=500, random_state=42)
    return -cross_val_score(m, X_tr, y_tr, cv=kf, scoring="r2").mean()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

p = study.best_params
m_optuna = MLPRegressor(hidden_layer_sizes=(p["h1"], p["h2"]),
                         activation=p["activation"], solver=p["solver"],
                         alpha=p["alpha"], learning_rate_init=p["lr"],
                         max_iter=500, random_state=42)
m_optuna.fit(X_tr, y_tr)
eval_reg(m_optuna, "sklearn Optuna", p["solver"])


sklearn Optuna                           | train R2=0.889 | test R2=0.868


{'train': {'R2': 0.8892903544869116,
  'MSE': 4591362.746953272,
  'MAE': 1349.909445602216},
 'test': {'R2': 0.8678973357298505,
  'MSE': 5342137.59682656,
  'MAE': 1413.23790662613}}

### 3.2 RandomizedSearchCV

In [7]:
param_dist = {
    "hidden_layer_sizes": [(64,32),(128,64),(256,128),(128,64,32)],
    "activation": ["relu","tanh"],
    "solver": ["adam","sgd","lbfgs"],
    "alpha": uniform(1e-5, 1e-2),
    "learning_rate_init": uniform(1e-4, 1e-2),
}
rs = RandomizedSearchCV(MLPRegressor(max_iter=500, random_state=42),
                         param_dist, n_iter=15, cv=kf,
                         scoring="r2", random_state=42, n_jobs=-1)
rs.fit(X_tr, y_tr)
eval_reg(rs.best_estimator_, "sklearn RandomSearch", rs.best_params_["solver"])


sklearn RandomSearch                     | train R2=0.926 | test R2=0.875


{'train': {'R2': 0.9260168030851583,
  'MSE': 3068239.3809593944,
  'MAE': 1157.0393555759604},
 'test': {'R2': 0.8753583035573712,
  'MSE': 5040421.375126417,
  'MAE': 1345.2499569545241}}

### 3.3 Hyperopt

In [ ]:
space = {
    "h1": hp.choice("h1", [64, 128, 256]),
    "h2": hp.choice("h2", [32, 64, 128]),
    "activation": hp.choice("activation", ["relu","tanh"]),
    "solver": hp.choice("solver", ["adam","sgd","lbfgs"]),
    "alpha": hp.loguniform("alpha", np.log(1e-5), np.log(1e-2)),
    "lr": hp.loguniform("lr", np.log(1e-4), np.log(1e-2)),
}

def hyperopt_obj(params):
    m = MLPRegressor(hidden_layer_sizes=(params["h1"], params["h2"]),
                     activation=params["activation"], solver=params["solver"],
                     alpha=params["alpha"], learning_rate_init=params["lr"],
                     max_iter=500, random_state=42)
    return {"loss": -cross_val_score(m, X_tr, y_tr, cv=3, scoring="r2").mean(),
            "status": STATUS_OK}

best = fmin(hyperopt_obj, space, algo=tpe.suggest, max_evals=10,
            trials=Trials(), rstate=np.random.default_rng(42))

solvers = ["adam","lbfgs"]
activations = ["relu","tanh"]
sizes = [64,128,256]; sizes2 = [32,64,128]
m_hp = MLPRegressor(hidden_layer_sizes=(sizes[best["h1"]], sizes2[best["h2"]]),
                     activation=activations[best["activation"]],
                     solver=solvers[best["solver"]],
                     alpha=best["alpha"], learning_rate_init=best["lr"],
                     max_iter=500, random_state=42)
m_hp.fit(X_tr, y_tr)
eval_reg(m_hp, "sklearn Hyperopt", solvers[best["solver"]])


  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

## 4. Keras + TensorFlow

In [ ]:
N = X_tr.shape[1]
EPOCHS = 30
CB = [keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]

def build(units1=64, units2=32, optimizer="adam", lr=1e-3):
    m = keras.Sequential([
        layers.Input(shape=(N,)),
        layers.Dense(units1, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(units2, activation="relu"),
        layers.Dense(1)
    ])
    opt = {"adam": keras.optimizers.Adam(lr),
           "sgd":  keras.optimizers.SGD(lr),
           "rmsprop": keras.optimizers.RMSprop(lr)}[optimizer]
    m.compile(optimizer=opt, loss="mse", metrics=["mae"])
    return m

def plot_history(h, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h.history["loss"], label="train")
    axes[0].plot(h.history["val_loss"], label="val")
    axes[0].set_title(f"{title} — Loss"); axes[0].legend()
    axes[1].plot(h.history["mae"], label="train")
    axes[1].plot(h.history["val_mae"], label="val")
    axes[1].set_title(f"{title} — MAE"); axes[1].legend()
    plt.tight_layout(); plt.show()

def eval_keras(model, label, optimizer):
    p_tr = model.predict(X_tr, verbose=0).flatten()
    p_te = model.predict(X_te, verbose=0).flatten()
    r = {"train": {"R2": r2_score(y_tr, p_tr),
                   "MSE": mean_squared_error(y_tr, p_tr),
                   "MAE": mean_absolute_error(y_tr, p_tr)},
         "test":  {"R2": r2_score(y_te, p_te),
                   "MSE": mean_squared_error(y_te, p_te),
                   "MAE": mean_absolute_error(y_te, p_te)}}
    print(f"{label:40s} | train R2={r['train']['R2']:.3f} | test R2={r['test']['R2']:.3f}")
    results.append({"Модель": label, "Оптимизатор": optimizer,
                    **{f"train_{k}": v for k, v in r["train"].items()},
                    **{f"test_{k}": v for k, v in r["test"].items()}})


NameError: name 'X_tr' is not defined

### 4.1 Базовые модели (Adam / SGD / RMSprop)

In [ ]:
for opt_name in ["adam", "sgd", "rmsprop"]:
    m = build(optimizer=opt_name)
    h = m.fit(X_tr, y_tr, validation_split=0.1,
              epochs=EPOCHS, callbacks=CB, verbose=0)
    plot_history(h, f"Keras {opt_name}")
    eval_keras(m, f"Keras {opt_name}", opt_name)
    m.summary()


### 4.2 Optuna

In [ ]:
def obj_kr(trial):
    m = build(trial.suggest_categorical("u1", [64,128,256]),
              trial.suggest_categorical("u2", [32,64,128]),
              trial.suggest_categorical("opt", ["adam","rmsprop"]),
              trial.suggest_float("lr", 1e-4, 1e-2, log=True))
    h = m.fit(X_tr, y_tr, validation_split=0.1, epochs=20, callbacks=CB, verbose=0)
    return min(h.history["val_loss"])

study_k = optuna.create_study(direction="minimize")
study_k.optimize(obj_kr, n_trials=8)
p = study_k.best_params
m_ko = build(p["u1"], p["u2"], p["opt"], p["lr"])
h = m_ko.fit(X_tr, y_tr, validation_split=0.1, epochs=EPOCHS, callbacks=CB, verbose=0)
plot_history(h, "Keras Optuna")
eval_keras(m_ko, "Keras Optuna", p["opt"])
m_ko.summary()


### 4.3 KerasTuner

In [ ]:
def kt_model(hp):
    return build(hp.Choice("u1", [64,128,256]),
                 hp.Choice("u2", [32,64,128]),
                 hp.Choice("opt", ["adam","rmsprop"]),
                 hp.Float("lr", 1e-4, 1e-2, sampling="log"))

tuner = kt.RandomSearch(kt_model, objective="val_loss", max_trials=10, seed=42,
                         directory="kt_reg", project_name="lab7_reg", overwrite=True)
tuner.search(X_tr, y_tr, validation_split=0.1, epochs=50, callbacks=CB, verbose=0)
best_m = tuner.get_best_models(1)[0]
best_hp = tuner.get_best_hyperparameters(1)[0]
h = best_m.fit(X_tr, y_tr, validation_split=0.1, epochs=EPOCHS, callbacks=CB, verbose=0)
plot_history(h, "KerasTuner")
eval_keras(best_m, "KerasTuner", best_hp.get("opt"))
best_m.summary()


### 4.4 Ray Tune

In [ ]:
ray.init(ignore_reinit_error=True, log_to_driver=False)

def ray_train(config):
    m = build(config["u1"], config["u2"], config["opt"], config["lr"])
    h = m.fit(X_tr, y_tr, validation_split=0.1, epochs=50, callbacks=CB, verbose=0)
    ray_tune.report({"val_loss": min(h.history["val_loss"])})

analysis = ray_tune.run(
    ray_train,
    config={"u1": ray_tune.choice([64,128,256]),
            "u2": ray_tune.choice([32,64,128]),
            "opt": ray_tune.choice(["adam","rmsprop"]),
            "lr": ray_tune.loguniform(1e-4, 1e-2)},
    search_alg=OptunaSearch(metric="val_loss", mode="min"),
    num_samples=5, verbose=0)

cfg = analysis.get_best_config("val_loss", "min")
print("Лучший конфиг:", cfg)
m_ray = build(cfg["u1"], cfg["u2"], cfg["opt"], cfg["lr"])
h = m_ray.fit(X_tr, y_tr, validation_split=0.1, epochs=EPOCHS, callbacks=CB, verbose=0)
plot_history(h, "Ray Tune")
eval_keras(m_ray, "Ray Tune", cfg["opt"])
m_ray.summary()


## 5. Сводная таблица метрик

In [ ]:
df_res = pd.DataFrame(results).round(4)
display(df_res)


## 6. Деплой — пример предсказания

In [ ]:
sample = X_te[[0]]
pred = m_optuna.predict(sample)[0]
print(f"Предсказанная цена: {pred:,.0f} USD")
print(f"Реальная цена:      {y_te[0]:,.0f} USD")


## 7. Вывод

Для задачи регрессии (предсказание цены автомобиля) были обучены FCNN двумя способами: через sklearn MLPRegressor и через Keras+TF.

Подбор гиперпараметров выполнялся тремя методами: Optuna, RandomizedSearchCV, Hyperopt (sklearn) и Optuna, KerasTuner, Ray Tune (Keras). Лучший оптимизатор — `adam`, как наиболее адаптивный.

По сравнению с LightGBM из lab4 (R²≈0.909), нейронные сети на табличных данных показывают сопоставимый, но чуть более слабый результат. Для улучшения можно увеличить глубину сети или добавить больше эпох обучения.
